In [19]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [20]:
SOURCE_DIR = Path("src")
OUTPUT_DIR = Path("out")
OUTPUT_DIR.mkdir(exist_ok=True)

BY_ARTICLE_PATH = SOURCE_DIR / "by_article.csv"
DECISION_TYPE_PATH = SOURCE_DIR / "decision_type.csv"
SANCTION_TYPE_PATH = SOURCE_DIR / "sanction_type.csv"
YEAR_TOTALS_PATH = SOURCE_DIR / "year_totals.csv"

for path in [BY_ARTICLE_PATH, DECISION_TYPE_PATH, SANCTION_TYPE_PATH, YEAR_TOTALS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")

In [21]:
by_article = pd.read_csv(BY_ARTICLE_PATH)
decision_type = pd.read_csv(DECISION_TYPE_PATH)
sanction_type = pd.read_csv(SANCTION_TYPE_PATH)
year_totals = pd.read_csv(YEAR_TOTALS_PATH)

by_article["Wert_Anzahl"] = pd.to_numeric(by_article["Wert_Anzahl"], errors="coerce")
decision_type["Wert_Anzahl"] = pd.to_numeric(decision_type["Wert_Anzahl"], errors="coerce")
sanction_type["Wert_Anzahl"] = pd.to_numeric(sanction_type["Wert_Anzahl"], errors="coerce")
year_totals["Wert_Anzahl"] = pd.to_numeric(year_totals["Wert_Anzahl"], errors="coerce")

In [22]:
year_summary = (
    year_totals.pivot_table(
        index="source_year",
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
    .sort_values("source_year")
)

year_summary["verurteilungsquote"] = year_summary["verurteilte"] / year_summary["abgeurteilte"]
year_summary["verurteilungsquote_pct"] = year_summary["verurteilungsquote"] * 100
year_summary[["abgeurteilte", "verurteilte"]] = year_summary[["abgeurteilte", "verurteilte"]].astype(int)

year_summary

Personengruppe,source_year,abgeurteilte,verurteilte,verurteilungsquote,verurteilungsquote_pct
0,2022,791090,647374,0.818332,81.833167
1,2023,804410,656901,0.816625,81.662461
2,2024,781610,632115,0.808735,80.873454


In [23]:
article_base = by_article.loc[
    (by_article["Geschlecht"] == "Insgesamt")
    & (by_article["Altersgruppe"] == ".")
    & (by_article["Angewandtes_Strafrecht"] == ".")
    & (by_article["Personengruppe"].isin(["Abgeurteilte insgesamt", "Verurteilte insgesamt"]))
].copy()

article_summary = (
    article_base.pivot_table(
        index=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
)

article_summary = article_summary.dropna(subset=["abgeurteilte", "verurteilte"])
article_summary = article_summary.loc[article_summary["abgeurteilte"] > 0].copy()
article_summary[["abgeurteilte", "verurteilte"]] = article_summary[["abgeurteilte", "verurteilte"]].astype(int)
article_summary["verurteilungsquote"] = article_summary["verurteilte"] / article_summary["abgeurteilte"]
article_summary["verurteilungsquote_pct"] = article_summary["verurteilungsquote"] * 100

article_summary.sort_values(["source_year", "abgeurteilte"], ascending=[True, False]).head(20)

Personengruppe,source_year,Straftat_Gesetz,Art_der_Straftat,abgeurteilte,verurteilte,verurteilungsquote,verurteilungsquote_pct
464,2022,StGBoV,Straftaten ohne Straftaten im Straßenverkehr n...,475914,370989,0.779529,77.952949
477,2022,Straftat,Straftaten außerhalb des Strafgesetzbuches (St...,191045,167355,0.875998,87.599780
433,2022,StGB,"StGB §§ 142, 315 b bis d,316 sowie 222,229, §§...",185490,164153,0.884970,88.496954
12,2022,And BuG,Straftaten nach anderen Bundes- und Landesgese...,129686,112232,0.865413,86.541338
109,2022,StGB,"StGB 22. Abschnitt, §§ 263 bis 266 b Betrug un...",124664,103392,0.829365,82.936533
455,2022,StGB,Straftaten im Straßenverkehr nach dem StGB ins...,124131,109030,0.878346,87.834626
456,2022,StGB,Straftaten im Straßenverkehr nach dem StGB und...,109244,91355,0.836247,83.624730
105,2022,StGB,"StGB 19. Abschnitt, §§ 242 bis 248 c Diebstahl...",106435,86164,0.809546,80.954573
264,2022,StGB,StGB § 242 Diebstahl,79688,65244,0.818743,81.874310
454,2022,StGB,Straftaten im Straßenverkehr nach dem StGB in ...,76246,72798,0.954778,95.477796


In [24]:
YEAR_SUMMARY_OUTPUT = OUTPUT_DIR / "year_conviction_rates.csv"
ARTICLE_SUMMARY_OUTPUT = OUTPUT_DIR / "article_conviction_rates.csv"

year_summary.to_csv(YEAR_SUMMARY_OUTPUT, index=False)
article_summary.to_csv(ARTICLE_SUMMARY_OUTPUT, index=False)

print(f"Wrote {YEAR_SUMMARY_OUTPUT} ({len(year_summary)} rows)")
print(f"Wrote {ARTICLE_SUMMARY_OUTPUT} ({len(article_summary)} rows)")

Wrote out\year_conviction_rates.csv (3 rows)
Wrote out\article_conviction_rates.csv (1471 rows)


In [ ]:
stgb_article_summary = article_summary.loc[
    article_summary["Straftat_Gesetz"].astype(str).str.strip().isin(["StGB", "StGBoV"])
].copy()

stgb_year_base = article_summary.loc[
    (
        (article_summary["Straftat_Gesetz"].astype(str).str.strip() == "StGBoV")
        & article_summary["Art_der_Straftat"].astype(str).str.contains(
            "Straftaten ohne Straftaten im Straßenverkehr nach dem StGB insgesamt Summe",
            na=False,
        )
    )
    |
    (
        (article_summary["Straftat_Gesetz"].astype(str).str.strip() == "StGB")
        & article_summary["Art_der_Straftat"].astype(str).str.contains(
            "Straftaten im Straßenverkehr nach dem StGB insgesamt Summe",
            na=False,
        )
    )
].copy()

stgb_year_summary = stgb_year_base.groupby("source_year", as_index=False)[["abgeurteilte", "verurteilte"]].sum().sort_values("source_year")
stgb_year_summary["verurteilungsquote"] = stgb_year_summary["verurteilte"] / stgb_year_summary["abgeurteilte"]
stgb_year_summary["verurteilungsquote_pct"] = stgb_year_summary["verurteilungsquote"] * 100

stgb_year_summary

In [ ]:
STGB_YEAR_OUTPUT = OUTPUT_DIR / "stgb_year_conviction_rates.csv"
STGB_ARTICLE_OUTPUT = OUTPUT_DIR / "stgb_article_conviction_rates.csv"

stgb_year_summary.to_csv(STGB_YEAR_OUTPUT, index=False)
stgb_article_summary.to_csv(STGB_ARTICLE_OUTPUT, index=False)

print(f"Wrote {STGB_YEAR_OUTPUT} ({len(stgb_year_summary)} rows)")
print(f"Wrote {STGB_ARTICLE_OUTPUT} ({len(stgb_article_summary)} rows)")

In [ ]:
CHART_DIR = OUTPUT_DIR / "charts" / "stgb"
CHART_DIR.mkdir(parents=True, exist_ok=True)

def plot_conviction_pies_by_year(frame: pd.DataFrame, year: int, output_dir: Path, charts_per_row: int = 3) -> Path:
    year_frame = frame.loc[frame["source_year"] == year].copy()
    year_frame = year_frame.sort_values(["abgeurteilte", "Art_der_Straftat"], ascending=[False, True]).reset_index(drop=True)
    year_frame["nicht_verurteilte"] = year_frame["abgeurteilte"] - year_frame["verurteilte"]

    n_charts = len(year_frame)
    n_cols = charts_per_row
    n_rows = (n_charts + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, max(4 * n_rows, 4)), squeeze=False)
    colors = ["#2E8B57", "#D9D9D9"]

    for ax, (_, row) in zip(axes.flat, year_frame.iterrows()):
        values = [row["verurteilte"], row["nicht_verurteilte"]]
        ax.pie(
            values,
            labels=["Verurteilte", "Nicht verurteilte"],
            colors=colors,
            startangle=90,
            counterclock=False,
            autopct=lambda pct: f"{pct:.1f}%" if pct > 0 else "",
            wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
            textprops={"fontsize": 8},
        )
        title = f"{str(row['Art_der_Straftat']).strip()}\nAbgeurteilte: {row['abgeurteilte']:,} | Verurteilte: {row['verurteilte']:,}".replace(',', ' ')
        ax.set_title(title, fontsize=9)

    for ax in axes.flat[n_charts:]:
        ax.axis("off")

    fig.suptitle(f"StGB conviction share by article, {year}", fontsize=16)
    fig.tight_layout()
    output_path = output_dir / f"stgb_pies_{year}.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return output_path

chart_paths = []
for year in sorted(stgb_article_summary["source_year"].unique()):
    chart_paths.append(plot_conviction_pies_by_year(stgb_article_summary, year, CHART_DIR))

chart_paths

In [ ]:
for path in chart_paths:
    print(path)

In [ ]:
import html

stgb_decision_base = decision_type.loc[
    (decision_type["Geschlecht"] == "Insgesamt")
    & decision_type["Straftat_Gesetz"].astype(str).str.strip().isin(["StGB", "StGBoV"])
].copy()

stgb_sanction_base = sanction_type.loc[
    (sanction_type["Geschlecht"] == "Insgesamt")
    & sanction_type["Straftat_Gesetz"].astype(str).str.strip().isin(["StGB", "StGBoV"])
].copy()

freispruch_ohne = (
    stgb_decision_base.loc[
        stgb_decision_base["Art_d_Entscheidung"].astype(str).str.strip()
        == "Abgeurteilte mit anderen Entscheidungen - Freispruch ohne Maßregeln"
    ]
    .groupby(["source_year", "Straftat_Gesetz", "Art_der_Straftat"], as_index=False)["Wert_Anzahl"]
    .sum()
    .rename(columns={"Wert_Anzahl": "freispruch_ohne_massregeln"})
)

freiheitsstrafe = (
    stgb_sanction_base.loc[
        stgb_sanction_base["Art_d_Entscheidung"].astype(str).str.strip()
        == "nach der schwersten Sanktion - Freiheitsstrafe insgesamt"
    ]
    .groupby(["source_year", "Straftat_Gesetz", "Art_der_Straftat"], as_index=False)["Wert_Anzahl"]
    .sum()
    .rename(columns={"Wert_Anzahl": "freiheitsstrafe_gesamt"})
)

bewaehrung = (
    stgb_sanction_base.loc[
        stgb_sanction_base["Art_d_Entscheidung"].astype(str).str.strip()
        == "zu Freiheitsstrafe bzw. Strafarrest mit Bewährung - Insgesamt"
    ]
    .groupby(["source_year", "Straftat_Gesetz", "Art_der_Straftat"], as_index=False)["Wert_Anzahl"]
    .sum()
    .rename(columns={"Wert_Anzahl": "bewaehrung"})
)

geldstrafe = (
    stgb_sanction_base.loc[
        stgb_sanction_base["Art_d_Entscheidung"].astype(str).str.strip()
        == "nach der schwersten Sanktion - Geldstrafe insgesamt"
    ]
    .groupby(["source_year", "Straftat_Gesetz", "Art_der_Straftat"], as_index=False)["Wert_Anzahl"]
    .sum()
    .rename(columns={"Wert_Anzahl": "geldstrafe"})
)

stgb_outcome_profile = stgb_article_summary.merge(
    freispruch_ohne,
    on=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
    how="left",
).merge(
    freiheitsstrafe,
    on=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
    how="left",
).merge(
    bewaehrung,
    on=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
    how="left",
).merge(
    geldstrafe,
    on=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
    how="left",
)

for column in ["freispruch_ohne_massregeln", "freiheitsstrafe_gesamt", "bewaehrung", "geldstrafe"]:
    stgb_outcome_profile[column] = stgb_outcome_profile[column].fillna(0)

stgb_outcome_profile["freiheitsstrafe"] = (
    stgb_outcome_profile["freiheitsstrafe_gesamt"] - stgb_outcome_profile["bewaehrung"]
).clip(lower=0)

stgb_outcome_profile["sonstige"] = (
    stgb_outcome_profile["abgeurteilte"]
    - stgb_outcome_profile["freispruch_ohne_massregeln"]
    - stgb_outcome_profile["freiheitsstrafe"]
    - stgb_outcome_profile["bewaehrung"]
    - stgb_outcome_profile["geldstrafe"]
).clip(lower=0)

for column in ["freispruch_ohne_massregeln", "freiheitsstrafe", "bewaehrung", "geldstrafe", "sonstige"]:
    stgb_outcome_profile[column] = stgb_outcome_profile[column].round().astype(int)

HTML_REPORT_OUTPUT = OUTPUT_DIR / "stgb_article_pies_report.html"
STGB_OUTCOME_OUTPUT = OUTPUT_DIR / "stgb_article_outcome_profile.csv"
stgb_outcome_profile.to_csv(STGB_OUTCOME_OUTPUT, index=False)

def build_article_card(row: pd.Series) -> str:
    parts = [
        ("Свободен", row["freispruch_ohne_massregeln"], "#d9d9d9"),
        ("Лишение свободы", row["freiheitsstrafe"], "#8b1e3f"),
        ("Условный срок", row["bewaehrung"], "#f4a261"),
        ("Денежный штраф", row["geldstrafe"], "#2a9d8f"),
        ("Остальные", row["sonstige"], "#6d597a"),
    ]
    total = int(row["abgeurteilte"])
    offset = 0.0
    segments = []
    legend_rows = []
    for label, value, color in parts:
        pct = (value / total * 100) if total else 0
        if pct > 0:
            segments.append(f"{color} {offset:.4f}% {offset + pct:.4f}%")
        legend_rows.append(f"<li><span class='swatch' style='background:{color}'></span>{label}: <strong>{int(value):,}</strong> <em>({pct:.2f}%)</em></li>".replace(',', ' '))
        offset += pct
    gradient = ', '.join(segments) if segments else '#d9d9d9 0 100%'
    title = html.escape(str(row["Art_der_Straftat"]).strip())
    law = html.escape(str(row["Straftat_Gesetz"]).strip())
    abgeurteilte = f"{int(row['abgeurteilte']):,}".replace(',', ' ')
    verurteilte = f"{int(row['verurteilte']):,}".replace(',', ' ')
    return f"""
    <article class='card' data-title='{title.lower()}'>
      <div class='pie' style='background: conic-gradient({gradient});'></div>
      <h3>{title}</h3>
      <p class='law'>{law}</p>
      <p>Abgeurteilte: <strong>{abgeurteilte}</strong></p>
      <p>Verurteilte: <strong>{verurteilte}</strong></p>
      <ul class='legend'>
        {''.join(legend_rows)}
      </ul>
    </article>
    """

sections = []
for year in sorted(stgb_outcome_profile['source_year'].unique()):
    year_frame = stgb_outcome_profile.loc[stgb_outcome_profile['source_year'] == year].sort_values(['abgeurteilte', 'Art_der_Straftat'], ascending=[False, True])
    cards_html = '\n'.join(build_article_card(row) for _, row in year_frame.iterrows())
    sections.append(f"""
    <details class='year-section' open>
      <summary>{year}</summary>
      <div class='grid'>{cards_html}</div>
    </details>
    """)

html_report = f"""
<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>StGB Conviction Report</title>
  <style>
    :root {{ --bg: #f5f1e8; --card: #fffdf8; --ink: #1f1f1f; --muted: #6e6a62; --line: #d8d1c4; }}
    * {{ box-sizing: border-box; }}
    body {{ margin: 0; font-family: Georgia, 'Times New Roman', serif; background: linear-gradient(180deg, #efe7d8 0%, var(--bg) 100%); color: var(--ink); }}
    .page {{ max-width: 1600px; margin: 0 auto; padding: 24px; }}
    .toolbar {{ position: sticky; top: 0; z-index: 10; padding: 16px 0; background: rgba(245,241,232,0.95); backdrop-filter: blur(8px); }}
    .toolbar input {{ width: min(520px, 100%); padding: 12px 14px; border: 1px solid var(--line); border-radius: 999px; font-size: 16px; background: white; }}
    h1 {{ margin: 0 0 8px; font-size: 36px; }}
    .intro {{ margin: 0 0 24px; color: var(--muted); font-size: 18px; }}
    .year-section {{ margin: 28px 0 16px; border-top: 2px solid var(--line); padding-top: 18px; }}
    summary {{ cursor: pointer; font-size: 28px; font-weight: 700; list-style: none; margin-bottom: 16px; }}
    summary::-webkit-details-marker {{ display: none; }}
    summary::before {{ content: '▾ '; }}
    details:not([open]) summary::before {{ content: '▸ '; }}
    .grid {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; }}
    .card {{ background: var(--card); border: 1px solid var(--line); border-radius: 18px; padding: 18px; box-shadow: 0 10px 30px rgba(60,40,10,0.06); min-height: 360px; }}
    .pie {{ width: 180px; height: 180px; border-radius: 50%; margin: 0 auto 16px; border: 10px solid #fff; }}
    h3 {{ margin: 0 0 8px; font-size: 20px; line-height: 1.2; }}
    .law {{ margin: 0 0 12px; color: var(--muted); font-size: 14px; letter-spacing: 0.04em; text-transform: uppercase; }}
    p {{ margin: 6px 0; line-height: 1.35; }}
    .legend {{ margin: 12px 0 0; padding: 0; list-style: none; }}
    .legend li {{ display: flex; align-items: center; gap: 8px; margin: 6px 0; font-size: 14px; }}
    .legend em {{ color: var(--muted); font-style: normal; }}
    .swatch {{ width: 12px; height: 12px; border-radius: 50%; display: inline-block; flex: 0 0 12px; }}
    .hidden {{ display: none !important; }}
    @media (max-width: 1200px) {{ .grid {{ grid-template-columns: repeat(2, minmax(0, 1fr)); }} }}
    @media (max-width: 800px) {{ .grid {{ grid-template-columns: 1fr; }} .page {{ padding: 16px; }} }}
  </style>
</head>
<body>
  <div class='page'>
    <div class='toolbar'><input id='search' type='search' placeholder='Search by article name'></div>
    <h1>StGB Outcomes by Article</h1>
    <p class='intro'>Each full circle is the full number of Abgeurteilte for one article. The slices show free acquittal, imprisonment, suspended imprisonment, fine, and all remaining outcomes.</p>
    {''.join(sections)}
  </div>
  <script>
    const search = document.getElementById('search');
    const sections = Array.from(document.querySelectorAll('.year-section'));
    const cards = Array.from(document.querySelectorAll('.card'));
    search.addEventListener('input', () => {{
      const query = search.value.trim().toLowerCase();
      cards.forEach(card => card.classList.toggle('hidden', query && !card.dataset.title.includes(query)));
      sections.forEach(section => section.classList.toggle('hidden', !section.querySelector('.card:not(.hidden)')));
    }});
  </script>
</body>
</html>
"""

HTML_REPORT_OUTPUT.write_text(html_report, encoding='utf-8')
HTML_REPORT_OUTPUT